# Data Retrieval — Reddit

| Field | Details |
|---|---|
| **Time window** | Jun 2024 – Nov 2024 |
| **Source** | Reddit — JSONL files (Pushshift / Academic Data Access) |
| **Collection** | Pre-downloaded before Reddit API shutdown by external contributor; not collected by this team |
| **Subreddits** | r/conservative · r/democrats · r/republican · r/worldnews · r/politics · r/liberal · r/trump |
| **Method** | Merge all per-subreddit JSONL files into two complete raw parquet files (comments + posts) — no cleaning or filtering applied |
| **Saved to** | `Data/1_Bronze/Reddit/reddit_comments_raw.parquet` · `Data/1_Bronze/Reddit/reddit_posts_raw.parquet` |

In [1]:
import json
from pathlib import Path
import pandas as pd

# Bronze directory holding all raw JSONL files (one per subreddit per type)
BRONZE_DIR = Path('../Data/1_Bronze/Reddit')

SUBREDDITS = ['conservative', 'democrats', 'republican', 'worldnews', 'politics', 'liberal', 'trump']

In [ ]:
# Load all comment JSONL files and merge into one DataFrame.
# Each comment retains link_id (-> root post) and parent_id (-> direct parent).
# subreddit_source is added explicitly in case the native 'subreddit' field is missing.
comment_frames = []

for sub in SUBREDDITS:
    path = BRONZE_DIR / f'r_{sub}_comments.jsonl'
    if not path.exists():
        print(f'MISSING: {path.name}')
        continue
    rows = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    df_sub = pd.DataFrame(rows)
    df_sub['subreddit_source'] = sub
    comment_frames.append(df_sub)
    print(f'r/{sub}: {len(df_sub):,} comments  |  columns: {df_sub.shape[1]}')

df_comments = pd.concat(comment_frames, ignore_index=True)
print(f'\nTotal comments: {df_comments.shape[0]:,} rows x {df_comments.shape[1]} columns')

r/conservative: 693,340 comments  |  columns: 73
r/democrats: 444,280 comments  |  columns: 74
r/republican: 93,495 comments  |  columns: 73


In [ ]:
# Load all post JSONL files and merge into one DataFrame.
# Posts do not have link_id/parent_id; they are the root documents.
# subreddit_source is added for the same reason as above.
post_frames = []

for sub in SUBREDDITS:
    path = BRONZE_DIR / f'r_{sub}_posts.jsonl'
    if not path.exists():
        print(f'MISSING: {path.name}')
        continue
    rows = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    df_sub = pd.DataFrame(rows)
    df_sub['subreddit_source'] = sub
    post_frames.append(df_sub)
    print(f'r/{sub}: {len(df_sub):,} posts  |  columns: {df_sub.shape[1]}')

df_posts = pd.concat(post_frames, ignore_index=True)
print(f'\nTotal posts: {df_posts.shape[0]:,} rows x {df_posts.shape[1]} columns')

In [ ]:
# Quick volume check per subreddit and verify key linkage columns are present
print('=== Comments ===')
print(df_comments['subreddit_source'].value_counts())
print(f'\nlink_id present: {"link_id" in df_comments.columns}')
print(f'parent_id present: {"parent_id" in df_comments.columns}')

print('\n=== Posts ===')
print(df_posts['subreddit_source'].value_counts())

In [ ]:
# Overview of combined DataFrames — key columns, dtypes, sample rows

KEY_COMMENT_COLS = ['subreddit_source', 'id', 'parent_id', 'link_id', 'author', 'body', 'score', 'created_utc']
KEY_POST_COLS    = ['subreddit_source', 'id', 'author', 'title', 'selftext', 'score', 'num_comments', 'upvote_ratio', 'created_utc']

print('=== Comments — key columns ===')
print(df_comments[[c for c in KEY_COMMENT_COLS if c in df_comments.columns]].dtypes.to_string())
print(f'\nSample:')
df_comments[[c for c in KEY_COMMENT_COLS if c in df_comments.columns]].head(3)

In [ ]:
print('=== Posts — key columns ===')
print(df_posts[[c for c in KEY_POST_COLS if c in df_posts.columns]].dtypes.to_string())
print(f'\nSample:')
df_posts[[c for c in KEY_POST_COLS if c in df_posts.columns]].head(3)

In [ ]:
# Save raw combined files to Bronze
out_comments = BRONZE_DIR / 'reddit_comments_raw.parquet'
out_posts    = BRONZE_DIR / 'reddit_posts_raw.parquet'

df_comments.to_parquet(out_comments, index=False)
df_posts.to_parquet(out_posts, index=False)

print(f'Saved:')
print(f'  {out_comments}  ({df_comments.shape[0]:,} rows x {df_comments.shape[1]} columns)')
print(f'  {out_posts}  ({df_posts.shape[0]:,} rows x {df_posts.shape[1]} columns)')